# 15 - accDM daughter: does a universal fluid closure exist?

Decide **A/B/C** (one formula in `x=k/k_fs` / f-dependent coefficients / interpolation tables) **before** fitting anything. We read the exact daughter response pointwise and test whether the effective sound speed and the shear closure each collapse onto one function of `x=k/k_fs` across `(tau, f, eta)`, over `k <= 1 Mpc^-1`, `f_acc <= 0.3`.

Spec: `docs/superpowers/specs/2026-07-05-fluid-closure-feasibility-diagnostic.md`. Plan: `docs/superpowers/plans/2026-07-05-fluid-closure-feasibility-diagnostic.md`. Style follows notebook 7.

**Targets (pole-masked, well-conditioned).** The first run of this notebook normalized by `ca2` and used `w_sigma`; both failed - `ca2 -> 0` for the cold daughter blew the ratio up to 1e6, and `w_sigma`/`w_theta` are identically zero in the *exact* hierarchy (leftover fluid-only scaffolding: `rho_plus_p_sshear` is never accumulated in the exact output loop, `perturbations.c:8837`). This version uses the raw quantities the fluid closure actually needs:
- **effective sound speed** `ceff2 = delta_p/delta_rho` (from `cs2_ncdm[1]`), raw, not divided by `ca2`;
- **shear closure** `sigma/delta` (the cvis2 sector);
- **kinematic cross-check** `R_v = k*sigma/theta`.

All three are ratios that **diverge where the denominator crosses zero** in free-streaming; we drop the smallest `drop_frac` of each denominator's magnitudes (`mask_small_denom`) before enveloping.

In [ ]:
import sys; sys.path.insert(0, '.')          # import helpers from notebooks_test/
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from classy import Class
from fluid_closure_helpers import (
    ca2_from_kfs, mask_small_denom, log_upper_envelope, collapse_band,
)

plt.rcParams.update({
    'mathtext.fontset': 'stix', 'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'legend.fontsize': 9, 'lines.linewidth': 1.5, 'figure.dpi': 200})
qual_colors = ['#377eb8', '#ff7f00', '#4daf4a', '#f781bf', '#984ea3']

# --- base cosmology + accDM model (same setup as notebook 7) ---
omega_b, omega_cdm0 = 0.022383, 0.12011
A_s, n_s, tau_reio, H0 = 2.1005829616811546e-9, 0.96605, 0.0543, 67.32
base_params = {'omega_b': omega_b, 'omega_cdm': omega_cdm0, 'H0': H0,
               'A_s': A_s, 'n_s': n_s, 'tau_reio': tau_reio}
PREC = {'output': 'mPk', 'P_k_max_1/Mpc': 10.0, 'z_max_pk': 0.0,
        'evolver': 0, 'reionization_z_start_max': 80}

A_T, MASS, KAPPA = 0.13, 1e16, 6.0
A_REC = 1.0 / (1.0 + 1090.0)

# --- diagnostic sweep grids ---
K_GRID   = np.logspace(-2, 0.0, 18)          # 1/Mpc: sub-horizon, spans below to above k_fs
F_LIST   = [0.05, 0.1, 0.2, 0.3]             # daughter fraction up to the target
ETA_LIST = [0.1, 1.0]                          # production (cold) + one warmer boost
DROP_FRAC = 0.2                               # pole mask: drop smallest 20% of |denominator|

def eps_acc_of_eta(eta):
    e2 = eta*eta
    return -e2 + np.sqrt(e2*e2 + 4*e2*eta + 5*e2 + 2*eta) - 2*eta

def accdm_params(eta, f_acc=0.1, kappa=KAPPA):
    """Exact-hierarchy accDM params for a given boost eta and daughter fraction f_acc."""
    ocdm = omega_cdm0 * (1 + f_acc*(1 - A_REC**kappa)/(1 + (A_REC/A_T)**kappa))**(-1)
    p = dict(base_params); p.update(PREC)
    p.update({'omega_cdm': ocdm,
              'vary_Gamma_acc': 'yes', 'kappa_acc': kappa, 'a_t_acc': A_T,
              'f_acc': f_acc, 'eta_acc': eta,
              'm_acc_in_GeV': MASS, 'm_cdm_in_GeV': MASS,
              'N_ncdm': 2, 'deg_ncdm': '3, 1',
              'm_ncdm': '0.02, {:.6e}'.format(MASS*1e9),
              'T_ncdm': '0.71611, 1', 'ncdm_quadrature_strategy': '0, 4',
              'ncdm_N_momentum_bins': '15, 501', 'N_ur': 0.00441,
              'background_Nloga': 5001, 'gauge': 'synchronous',
              'get_perturbations_in_current_gauge': 'yes',
              'ncdm_fluid_trigger_tau_over_tau_k': 25,
              'ncdm_fluid_approximation': 3})          # 3 = none (exact hierarchy)
    return p

print('setup OK; W(eta=0.1) =', round(1 - 2*eps_acc_of_eta(0.1), 4),
      '| grids: k<=1 x{}, f x{}, eta x{}'.format(K_GRID.size, len(F_LIST), len(ETA_LIST)))

## Task 2 - tau-series extractor

Run the exact hierarchy once and keep the **full tau-series** of the daughter's `{delta, theta, shear, delta_p/delta_rho, k_fs}` for each requested k. `aH(tau)` is recovered from the **background** (`a*H` interpolated on conformal time), and base `ca2 = (3/2)(aH/k_fs)^2` (used only for the fit-prediction overlay).

In [ ]:
def _find_key(d, want):
    if want in d:
        return want
    for kk in d:
        if kk.replace(' ', '').startswith(want.replace(' ', '')):
            return kk
    raise KeyError('{!r} not found; available: {}'.format(want, list(d.keys())))

def extract_daughter_series(params, k_list):
    """Run exact CLASS; return {k: dict of daughter tau-series arrays}."""
    ks = np.sort(np.asarray(k_list, float))
    p = dict(params); p['k_output_values'] = ', '.join('{:.8e}'.format(k) for k in ks)
    M = Class(); M.set(p); M.compute()
    perts = M.get_perturbations()['scalar']            # one dict per k, sorted-k order
    bg = M.get_background()
    tau_bg = np.asarray(bg['conf. time [Mpc]'], float)
    a_bg   = 1.0 / (1.0 + np.asarray(bg['z'], float))
    H_bg   = np.asarray(bg['H [1/Mpc]'], float)
    o = np.argsort(tau_bg); tau_bg, a_bg, H_bg = tau_bg[o], a_bg[o], H_bg[o]
    kd  = _find_key(perts[0], 'delta_ncdm[1]'); kt  = _find_key(perts[0], 'theta_ncdm[1]')
    ksh = _find_key(perts[0], 'shear_ncdm[1]'); kc  = _find_key(perts[0], 'cs2_ncdm[1]')
    kf  = _find_key(perts[0], 'k_fss_acc[1]');  ktau = _find_key(perts[0], 'tau')
    out = {}
    for k, d in zip(ks, perts):
        tau = np.asarray(d[ktau], float)
        aH  = np.interp(tau, tau_bg, a_bg) * np.interp(tau, tau_bg, H_bg)
        out[k] = dict(tau=tau, aH=aH,
                      delta=np.asarray(d[kd], float), theta=np.asarray(d[kt], float),
                      shear=np.asarray(d[ksh], float), dpr=np.asarray(d[kc], float),
                      k_fs=np.asarray(d[kf], float))
    M.struct_cleanup(); M.empty()
    return out

In [ ]:
# structure-check: one cheap extraction, fail fast on a bad column name or a bad ca2
_probe = extract_daughter_series(accdm_params(ETA_LIST[0], f_acc=0.1), K_GRID[:3])
assert set(_probe.keys()) == set(K_GRID[:3])
_s = _probe[K_GRID[0]]
for q in ('theta', 'shear', 'dpr', 'k_fs', 'aH', 'tau'):
    assert _s[q].shape == _s['delta'].shape, q
g = (_s['k_fs'] > 0) & np.isfinite(_s['aH'])
ca2 = ca2_from_kfs(_s['k_fs'][g], _s['aH'][g])
print('extractor OK; tau samples per k =', _s['delta'].size,
      '| ca2 range = [{:.2e}, {:.2e}]'.format(ca2.min(), ca2.max()),
      '| x=k/k_fs reaches {:.0f}'.format((K_GRID.max()/_s['k_fs'][g]).max()))
assert np.all(ca2 <= 1.0 + 1e-6), 'ca2 > 1 -> aH/k_fs mismatch (check background keys)'

## Task 3 - assemble the pole-masked response envelopes across (f, eta)

- **`ceff2 = delta_p/delta_rho`** (raw), pole-masked at `delta ~ 0`. Overlaid with `ceff2_fit = ca2*(1 + 0.2*W*sqrt(x))` (what the paper form predicts) and the causal ceiling `1/3`.
- **`sig_del = sigma/delta`** (shear closure), pole-masked at `delta ~ 0`.
- **`Rv = k*sigma/theta`** (kinematic cross-check), pole-masked at `theta ~ 0`.

Each is enveloped (upper `|.|` per log-x bin) after masking. `x = k/k_fs` reaches ~1e3 even for `k <= 1` because `k_fs` is tiny - so the free-streaming regime (`x >> 1`) is unavoidably sampled.

In [ ]:
def build_responses(f, eta, drop_frac=DROP_FRAC):
    ser = extract_daughter_series(accdm_params(eta, f_acc=f), K_GRID)
    W = 1 - 2*eps_acc_of_eta(eta)
    Xc, Ce, Cf, Sd, Xv, Rv = [], [], [], [], [], []
    for k, s in ser.items():
        g = np.isfinite(s['k_fs']) & (s['k_fs'] > 0) & np.isfinite(s['aH'])
        if not np.any(g):
            continue
        x   = k / s['k_fs'][g]
        ca2 = ca2_from_kfs(s['k_fs'][g], s['aH'][g])
        delta = s['delta'][g]; theta = s['theta'][g]
        shear = s['shear'][g]; dpr = s['dpr'][g]
        safe_d = np.where(delta == 0, np.nan, delta)
        safe_t = np.where(theta == 0, np.nan, theta)
        Xc.append(x)
        Ce.append(np.abs(mask_small_denom(dpr, delta, drop_frac)))          # raw ceff2
        Cf.append(ca2 * (1 + 0.2*W*np.sqrt(x)))                             # paper-form prediction
        Sd.append(np.abs(mask_small_denom(shear/safe_d, delta, drop_frac)))  # sigma/delta
        Xv.append(x)
        Rv.append(np.abs(mask_small_denom(k*shear/safe_t, theta, drop_frac)))  # k*sigma/theta
    cat = np.concatenate
    return {'ceff2':     log_upper_envelope(cat(Xc), cat(Ce)),
            'ceff2_fit': log_upper_envelope(cat(Xc), cat(Cf)),
            'sig_del':   log_upper_envelope(cat(Xc), cat(Sd)),
            'Rv':        log_upper_envelope(cat(Xv), cat(Rv))}

RESPONSES = {}
for eta in ETA_LIST:
    for f in tqdm(F_LIST, desc='eta={}'.format(eta)):
        RESPONSES[(f, eta)] = build_responses(f, eta)
print('built', len(RESPONSES), 'responses:', list(RESPONSES.keys()))

## Task 4 - collapse plots + A/B/C decision

Left: raw `ceff2` vs the paper-form prediction and the `1/3` ceiling. Right: `sigma/delta`. Collapse quantified two ways: **pooled over all** `(f, eta)` and **within fixed f**. `TOL = 0.15`.

In [ ]:
X_EVAL = np.logspace(-1, 3, 60)
ls_eta = {ETA_LIST[0]: '-', ETA_LIST[1]: '--'}
col_f  = {f: qual_colors[i] for i, f in enumerate(F_LIST)}

def band_all(w):
    return collapse_band(X_EVAL, [RESPONSES[k][w] for k in RESPONSES])[1]
def band_fixed_f(w):
    return float(np.nanmax([collapse_band(X_EVAL, [RESPONSES[(f, e)][w] for e in ETA_LIST])[1]
                            for f in F_LIST]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
for (f, eta), r in RESPONSES.items():
    for ax, w in zip(axes, ('ceff2', 'sig_del')):
        x, env = r[w]
        if x.size:
            ax.loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.85,
                      label='f={}, eta={}'.format(f, eta))
xf, ef = RESPONSES[(0.1, 0.1)]['ceff2_fit']
axes[0].loglog(xf, ef, 'k:', label=r'paper-form $c_a^2(1{+}0.2W\sqrt{x})$')
axes[0].axhline(1./3., color='red', ls='--', lw=1.0, label=r'causal $1/3$')
axes[0].set_title(r'raw $c_{\rm eff}^2=\delta p/\delta\rho$ (pole-masked)')
axes[1].set_title(r'$\sigma/\delta$ (shear closure, pole-masked)')
for ax in axes:
    ax.set_xlabel(r'$x=k/k_{\rm fs}$'); ax.grid(True, which='both', alpha=0.3)
axes[0].legend(fontsize=7, ncol=2); plt.show()

band = {w: (band_all(w), band_fixed_f(w)) for w in ('ceff2', 'sig_del', 'Rv')}
print('band (pooled-all, fixed-f):')
for w in ('ceff2', 'sig_del', 'Rv'):
    print('  {:>8}: {:.3f}, {:.3f}'.format(w, *band[w]))
# how far is the paper-form prediction from the measured effective sound speed?
xm, em = RESPONSES[(0.1, 0.1)]['ceff2']
fit_on_xm = np.interp(np.log10(xm), np.log10(xf), ef)
ratio = em / fit_on_xm
print('measured/paper-form ceff2 (f=0.1): median {:.1f}x, max {:.1e}x'.format(
    np.nanmedian(ratio), np.nanmax(ratio)))

In [ ]:
TOL = 0.15
def verdict(w):
    pooled, fixedf = band[w]
    if not np.isfinite(pooled):
        return 'NO DATA'
    if pooled < TOL:
        return 'UNIVERSAL (one formula in x)'
    if np.isfinite(fixedf) and fixedf < TOL:
        return 'F-DEPENDENT (f-varying coefficients)'
    return 'NO COLLAPSE (interpolation tables / fluid inadequate)'

ve, vs = verdict('ceff2'), verdict('sig_del')
print('ceff2 (sound-speed sector):', ve)
print('sigma/delta (shear sector):', vs)

def sev(v):
    return 0 if 'UNIVERSAL' in v else (1 if 'F-DEP' in v else 2)
overall = max(sev(ve), sev(vs))
print('\nAPPROACH ->', ['B/A universal (one formula in x)',
                        'A with f-dependent coefficients',
                        'C tables / fluid inadequate -> exact + q(f) schedule (nb14)'][overall])

## Task 5 - where does collapse fail, and the kinematic cross-check

Left: the collapse band **as a function of x** for both sectors - if the fluid is only viable where the daughter clusters, the band stays under `TOL` at low `x` and blows past it for `x >> 1`. Right: the kinematic `R_v = k*sigma/theta` cross-check (independent of the `sigma/delta` probe).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), constrained_layout=True)
# (left) band(x): where does collapse hold?
bc, _ = collapse_band(X_EVAL, [RESPONSES[k]['ceff2']   for k in RESPONSES])
bs, _ = collapse_band(X_EVAL, [RESPONSES[k]['sig_del'] for k in RESPONSES])
axes[0].loglog(X_EVAL, bc, '-',  color=qual_colors[0], label=r'$c_{\rm eff}^2$ band$(x)$')
axes[0].loglog(X_EVAL, bs, '-',  color=qual_colors[2], label=r'$\sigma/\delta$ band$(x)$')
axes[0].axhline(TOL, color='red', ls='--', lw=1.0, label='TOL=0.15')
axes[0].axvline(1.0, color='gray', ls=':', lw=1.0, label=r'$x=1$ (free-streaming onset)')
axes[0].set_title('collapse band vs x (pooled over f, eta)')
axes[0].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[0].set_ylabel('fractional band width')
axes[0].grid(True, which='both', alpha=0.3); axes[0].legend(fontsize=8)
# (right) kinematic shear cross-check
for (f, eta), r in RESPONSES.items():
    x, env = r['Rv']
    if x.size:
        axes[1].loglog(x, env, ls_eta[eta], color=col_f[f], alpha=0.85,
                       label='f={}, eta={}'.format(f, eta))
axes[1].set_title(r'cross-check $R_v=k\sigma/\theta$ (pole-masked)')
axes[1].set_xlabel(r'$x=k/k_{\rm fs}$'); axes[1].grid(True, which='both', alpha=0.3)
axes[1].legend(fontsize=7, ncol=2)
plt.show()

print('R_v band (pooled, fixed-f): {:.3f}, {:.3f} | verdict:'.format(*band['Rv']), verdict('Rv'))
lowx = X_EVAL < 1.0
print('ceff2 band: max at x<1 = {:.3f}, max at x>1 = {:.3f}'.format(
    np.nanmax(bc[lowx]) if np.any(np.isfinite(bc[lowx])) else np.nan,
    np.nanmax(bc[~lowx]) if np.any(np.isfinite(bc[~lowx])) else np.nan))

## Verdict

*(Fill the bracketed numbers from the executed `band` dict and printed verdicts.)*

- **`ceff2` collapse:** pooled band = [FILL], fixed-f band = [FILL] -> **[UNIVERSAL / F-DEP / NO-COLLAPSE]**. Measured `ceff2` is [FILL]x the paper-form prediction (median) -> the multiplicative `ca2*(1+amp*W*sqrt(x))` ansatz is [salvageable / structurally wrong].
- **`sigma/delta` collapse:** pooled band = [FILL], fixed-f band = [FILL] -> **[UNIVERSAL / F-DEP / NO-COLLAPSE]**; `R_v` cross-check [agrees / disagrees].
- **Where collapse holds:** `ceff2` band < TOL for x [< / never <] 1; blows to [FILL] for x > 1. Because `k_fs` is tiny, `k <= 1` still reaches `x ~ 1e3`, so the free-streaming regime dominates.
- **Chosen approach:** [B/A universal | A with f-dependent coefficients | C tables / fluid inadequate].
- **Feasibility gate:** if the sound-speed sector does not collapse over the `x` range that `k <= 1` actually samples, no smooth `ceff2(x)` reaches ~1% P(k) -> route to **exact + q(f) schedule (notebook 14)** for `f` up to 0.3, per `memory: accdm-fluid-f-boundary`.
- **Next:** [write the fit plan matched to the verdict | consolidate nb14 as production]. Do NOT touch `perturbations_ceff2_ncdm` before that decision.

**Keep in sync:** `fluid_closure_helpers.py` is unit-tested in `test_fluid_closure_helpers.py`; `w_sigma`/`w_theta`/`w_p` columns are **zero in the exact hierarchy** (`perturbations.c:8835-8837`, moments never accumulated) - do not use them as exact-run probes.